Test 1: No duplicate appointments


In [0]:
%sql
SELECT
  appointment_id,
  COUNT(*) AS duplicate_count
FROM poc_health_no_show.silver.appointments_clean
GROUP BY appointment_id
HAVING COUNT(*) > 1;

Test 2: No invalid ages

In [0]:
%sql
SELECT COUNT(*) AS invalid_age_count
FROM poc_health_no_show.silver.appointments_clean
WHERE age < 0 OR age > 120;

Validate the override

In [0]:
%sql
SELECT
  appointment_id,
  age,
  risk_segment,
  risk_reason,
  risk_model_version
FROM poc_health_no_show.gold.appointment_risk_segments
WHERE risk_model_version = 'uc16_rules_v1.1_with_age_override';

In [0]:
%sql
SELECT COUNT(*) AS invalid_age_count
FROM poc_health_no_show.gold.appointment_risk_segments
WHERE age < 0 OR age > 120;

In [0]:
%sql
SELECT COUNT(*) AS total_appointments
FROM poc_health_no_show.gold.appointment_risk_segments;

Test 3: No invalid date sequence

In [0]:
%sql
SELECT COUNT(*) AS invalid_date_count
FROM poc_health_no_show.silver.appointments_clean
WHERE data_quality_status = 'SCHEDULED_AFTER_APPOINTMENT';

Test 4: Risk segment values are valid

In [0]:
%sql
SELECT DISTINCT risk_segment
FROM poc_health_no_show.gold.appointment_risk_segments
ORDER BY risk_segment;

Test 5: Risk score matches segment

In [0]:
%sql
SELECT COUNT(*) AS mismatch_count
FROM poc_health_no_show.gold.appointment_risk_segments
WHERE CASE
  WHEN risk_score >= 70 THEN risk_segment <> 'HIGH'
  WHEN risk_score >= 50 THEN risk_segment <> 'MEDIUM_HIGH'
  WHEN risk_score >= 30 THEN risk_segment <> 'MEDIUM'
  ELSE risk_segment <> 'LOW'
END;

Test 6: High-risk segment has higher no-show rate

In [0]:
%sql
SELECT
  risk_segment,
  ROUND(
    AVG(CASE WHEN no_show_flag THEN 1.0 ELSE 0.0 END) * 100,
    2
  ) AS no_show_rate_pct
FROM poc_health_no_show.gold.appointment_risk_segments
GROUP BY risk_segment
ORDER BY no_show_rate_pct DESC;

Test 7: AI messages are non-empty

In [0]:
%sql
SELECT COUNT(*) AS empty_message_count
FROM poc_health_no_show.gold.outreach_message_variants
WHERE message_text IS NULL OR trim(message_text) = '';

Test 8: AI messages do not contain unsafe terms

In [0]:
%sql
SELECT appointment_id, variant, message_text
FROM poc_health_no_show.gold.outreach_message_variants
WHERE lower(message_text) RLIKE
  'diagnos|treatment|medic|dose|disease|condition|prescri';

In [0]:
%sql
SELECT
  COUNT(*) AS same_day_appointments,
  SUM(CASE WHEN scheduled_ts > appointment_ts THEN 1 ELSE 0 END) AS flagged_by_timestamp_rule
FROM poc_health_no_show.silver.appointments_clean
WHERE CAST(scheduled_ts AS DATE) = CAST(appointment_ts AS DATE);

In [0]:
%sql
SELECT appointment_id, age, no_show_flag
FROM poc_health_no_show.silver.appointments_clean
WHERE age < 0 OR age > 120;

In [0]:
%sql
SELECT
  data_quality_status,
  COUNT(*) AS appointment_count
FROM poc_health_no_show.silver.appointments_clean
GROUP BY data_quality_status
ORDER BY appointment_count DESC;

In [0]:
%sql
SELECT
  MIN(lead_time_days) AS min_lead_time,
  MAX(lead_time_days) AS max_lead_time,
  COUNT(CASE WHEN lead_time_days < 0 THEN 1 END) AS negative_lead_time_count
FROM poc_health_no_show.silver.appointments_clean
WHERE data_quality_status = 'VALID';